In [0]:
%sql
USE CATALOG day8_catalog;
USE SCHEMA ecommerce;



In [0]:
%sql
CREATE OR REPLACE VIEW daily_revenue AS
SELECT
  DATE(event_time) AS event_date,
  SUM(price) AS revenue
FROM events_silver
WHERE event_type = 'purchase'
GROUP BY DATE(event_time);


In [0]:
%sql
SELECT * FROM daily_revenue ORDER BY event_date LIMIT 10;


In [0]:
%sql
SELECT
  event_date,
  revenue,
  AVG(revenue) OVER (
    ORDER BY event_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
  ) AS revenue_ma7
FROM daily_revenue
ORDER BY event_date;


In [0]:
%sql
CREATE OR REPLACE VIEW conversion_funnel AS
SELECT
  category_code,
  SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
  SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases,
  ROUND(
    SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) * 100.0 /
    NULLIF(SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END), 0),
    2
  ) AS conversion_rate
FROM events_silver
GROUP BY category_code;


In [0]:
%sql
SELECT * FROM conversion_funnel ORDER BY conversion_rate DESC;


In [0]:
%sql
CREATE OR REPLACE VIEW customer_tiers AS
WITH purchases AS (
  SELECT
    user_id,
    COUNT(*) AS purchase_count,
    SUM(price) AS total_spent
  FROM events_silver
  WHERE event_type = 'purchase'
  GROUP BY user_id
)
SELECT
  CASE
    WHEN purchase_count >= 10 THEN 'VIP'
    WHEN purchase_count >= 5 THEN 'Loyal'
    ELSE 'Regular'
  END AS customer_tier,
  COUNT(*) AS customers,
  ROUND(AVG(total_spent), 2) AS avg_ltv
FROM purchases
GROUP BY customer_tier;


In [0]:
%sql
SELECT * FROM customer_tiers;


In [0]:
%sql
SELECT
  brand,
  SUM(price) AS total_revenue
FROM events_silver
WHERE event_type = 'purchase'
GROUP BY brand
ORDER BY total_revenue DESC
LIMIT 10;
